In [1]:
import pandas as pd

In [2]:
# Load data
df = pd.read_csv("perfect_clean_dataset.csv")

In [3]:
# Clean and convert date
df["trade_date"] = (
    df["trade_date"]
    .astype("string")
    .str.strip()
)

df["trade_date"] = pd.to_datetime(
    df["trade_date"],
    errors="coerce",
    dayfirst=True
)

In [4]:
# Remove duplicates
df = df.drop_duplicates().reset_index(drop=True)

# Sort by ticker and trade_id
df = df.sort_values(
    ["ticker", "trade_id"]
).reset_index(drop=True)


In [10]:
# FILL MISSING DATES
# ============================================================

for ticker in df["ticker"].dropna().unique():

    ticker_rows = df["ticker"] == ticker

    valid_dates = (
        df.loc[
            ticker_rows & df["trade_date"].notna(),
            "trade_date"
        ]
        .sort_values()
        .drop_duplicates()
        .reset_index(drop=True)
    )

    missing_rows = df.index[
        ticker_rows & df["trade_date"].isna()
    ]

    if len(valid_dates) == 0 or len(missing_rows) == 0:
        continue

In [13]:
# FILL MISSING DATE USING TICKER + EXISTING DATE PATTERN
# ------------------------------------------------------------

df["trade_date"] = (
    df.groupby("ticker")["trade_date"]
      .transform(lambda x: x.ffill().bfill())
)

In [14]:
# FINAL CHECK
# ------------------------------------------------------------

print("Duplicate Rows:", df.duplicated().sum())

print("\nMissing Values:")
print(df.isna().sum())

print("\nMissing trade_date:")
print(df["trade_date"].isna().sum())

print("\nDate Range:")
print(df["trade_date"].min())
print(df["trade_date"].max())

Duplicate Rows: 0

Missing Values:
trade_id              0
ticker                0
company_name          0
exchange              0
sector                0
trade_date            0
open_price_inr        0
high_price_inr        0
low_price_inr         0
close_price_inr       0
prev_close_inr        0
volume                0
market_cap_cr         0
pe_ratio              0
pb_ratio              0
dividend_yield        0
beta                  0
52w_high_inr          0
52w_low_inr           0
nifty50_weight        0
portfolio             0
analyst_rating        0
shares_held           0
purchase_price_inr    0
currency              0
dtype: int64

Missing trade_date:
0

Date Range:
2020-01-01 00:00:00
2024-12-12 00:00:00


In [15]:
# SAVE CLEAN DATA
# ------------------------------------------------------------

df.to_csv(
    "Clean_portflio_data.csv",
    index=False
)